# MinSearch Vector

Minsearch::
1. minsearch is a memory search engine, if we trun off the laptop, then we lost all the documents and we need to load them again.

2. minsearch has a vector search also, we will implement it

source:  https://www.youtube.com/watch?v=E7KdO3xmESg&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=23

In [1]:
from pathlib import Path
from rich import print
import os
import sys
import numpy as np
from minsearch import VectorSearch



parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer
from langchain_core.documents import Document


# 1. load the documents
print('---------------------------------------------------------------')
print('Step 1: load the data')
print('---------------------------------------------------------------')
documents = load_faq_data()
print(f'The number of documents is: {len(documents)}')
print(documents[0])


print('---------------------------------------------------------------')
print('Step 2: Get the useful data only from each documant')
print('---------------------------------------------------------------')
# extract only the question and answer from each documents
texts = list()

for doc in documents:
    text_per_document = doc['question'] + " " + doc['answer']
    texts.append(text_per_document)

print(f'The number of text documents is: {len(texts)}')
print(texts[0])

---------------------------------------------------------------

Step 1: load the data

---------------------------------------------------------------

The number of documents is: 1350

{
    'id': '9e508f2212',
    'course': 'data-engineering-zoomcamp',
    'section': 'General Course-Related Questions',
    'question': 'Course: When does the course start?',
    'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and 
registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n-
Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram 
channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's 
[Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."
}

---------------------------------------------------------------

Step 2: Get the useful data only from each documant

---------------------------------------------------------------

The number of text documents is: 1350

Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's 
exact start date and registration link, check the (https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the (https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` 
channel.

In [3]:
print(texts[0])

Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's 
exact start date and registration link, check the (https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the (https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` 
channel.

In [ ]:
print('---------------------------------------------------------------')
print('Step 3: chunck the data')
print('---------------------------------------------------------------')
# why we chunck
# improves retrieval accuracy,
# gives the LLM less irrelevant context,
# reduces token usage.
# 3.1. Convert your list of text strings into a list of LangChain Document objects
# This keeps your 1,350 documents separate and un-merged.
langchain_docs = [Document(page_content=text) for text in texts]


# Note::: I changed the chunck_size to 5000 so that I get the same numbers of vectors and documents to overcome an error
# 3.2. Setup the Recursive Text Splitter
# It chunks your text smartly, keeping paragraphs/sentences together, and creates overlaps!
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=5000,    # A high number ensures a single Q&A text never gets split up
    # chunk_size=500, # for improvement
    chunk_overlap=0,    # Zero overlap ensures no repeated data, just like your old loop
    # chunk_overlap=50, # for improvement
    # length_function=lambda text: len(tokenizer.encode(text)) # Counting tokens, # for improvement and by default length_function=len
    # separators=[r"Question \d+:", "\n\n", "\n", " "],# for improvement. note custome seperator= separators=["\n===\n", "\n\n", "\n", " "] but since I have only question and answer, I set it differently
    # is_separator_regex=True # for improvement 
)

# split the docs (chuncks)
texts_chuncks = text_splitter.split_documents(langchain_docs) # Note: we do not use  text_splitter.split_text(" ".join(texts))  since we have a list of document, each document will be splitted into chuncks, then langchain merges all the chunck

print(f"The number of chunks documents is: {len(texts_chuncks)}")
print(texts_chuncks[0])


---------------------------------------------------------------

Step 3: chunck the data

---------------------------------------------------------------

The number of chunks documents is: 1350

Document(
    metadata={},
    page_content="Course: When does the course start? A new cohort runs roughly January–April every year. For the 
current cohort's exact start date and registration link, check the [course repo 
README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo 
before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join
DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."
)

In [8]:
print('---------------------------------------------------------------')
print('Step 4: embed the data')
print('---------------------------------------------------------------')
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-V2", # same model as in datatalk
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True} # this is for normalizing (sumation =1 of a vector)
)
vectors = embeddings.embed_documents([doc.page_content for doc in texts_chuncks ])

# cast the list to array
array_vectors = np.array(vectors)

print(f'the number of vectors is: {len(array_vectors)}')

---------------------------------------------------------------

Step 4: embed the data

---------------------------------------------------------------

the number of vectors is: 1350

# Given vectors, create an index vector by minsearch

In [9]:
print('---------------------------------------------------------------')
print('Step 5: create an index and fit the data')
print('---------------------------------------------------------------')
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(array_vectors, documents)

---------------------------------------------------------------

Step 5: create an index and fit the data

---------------------------------------------------------------

# Test a vectorquery by minsearch

In [10]:
query = "I just discovered the course. Can I still join it?"
query_vector = embeddings.embed_query(query)

# get the result instead of computing similary by yourself, use minsearch
results = vindex.search(np.array(query_vector), num_results=5)

# look at the first 
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

# filter by course (filter, then search the top 5)

In [11]:
results = vindex.search(np.array(query_vector),
                        filter_dict={"course": "llm-zoomcamp"},
                        num_results=5)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co